In [2]:
import pickle
from random import randint, random, choice
from game import Game, Move, Player
from tqdm.auto import tqdm
from tqdm import trange
from typing import Literal
import numpy as np

## Random Player Definition

In [3]:
class RandomPlayer(Player):
    def __init__(self) -> None:
        super().__init__()

    def choose_action(self, game: 'Game') -> tuple[tuple[int, int], Move]:
        from_pos = (randint(0, 4), randint(0, 4))
        move = choice([Move.TOP, Move.BOTTOM, Move.LEFT, Move.RIGHT])
        return from_pos, move

    def give_rew(self, reward):
        pass

    def add_state(self, s):
        pass

## Reinforcement Learning Player

In [ ]:
class RLPlayer(Player):
    def __init__(self, epochs: int,
                 alpha: float,
                 discount_factor: float,
                 min_exploration_rate: float,
                 exploration_decay_rate: float,
                 opponent: 'Player') -> None:
        
        super().__init__()
        self.epochs = epochs
        self.alpha = alpha
        self.discount_factor = discount_factor
        self.exploration_rate = 1
        self.min_exploration_rate = min_exploration_rate
        self.exploration_decay_rate = exploration_decay_rate
        self.opponent = opponent
        self.states=[]
        self.state_value = {}
    
    def give_rew(self, reward):
        pass
    
    def add_state(self, state):
        self.states.append(state)
        
    def reset(self):
        self.states = []
    
    def choose_action(self, game: Game) -> tuple[tuple[int, int], Move]:
        available_moves = game.get_possible_moves(game.get_current_player())
        if random() < self.exploration_rate:  # do exploration for 30% of the time
            return choice(available_moves)
        else:  # take the best one for 70% of the time
            value_max = -999
            for move in available_moves:
                tmp = game.get_board()
                game.make_move(move[0], move[1])
                next_status = game.convert_matrix_board_to_tuple(game.get_board())
                game.set_board(tmp)
                value = 0 if self.state_value.get(next_status) is None else self.state_value.get(next_status)
                if value > value_max:
                    value_max = value
                    action = move
        return action

    def update_state_value_table(self, state_after_move, reward):
        if state_after_move not in self.state_value:
            self.state_value[state_after_move] = 0
        current_q_value = self.state_value[state_after_move]
        self.state_value[state_after_move] = current_q_value + self.alpha * (self.discount_factor * reward - current_q_value)
        
    def update_state_value_table2(self, reward):
        for st in reversed(self.states):
            if self.state_value.get(st) is None:
                self.state_value[st] = 0
            current_q_value = self.state_value[st]
            reward = current_q_value + self.alpha * (self.discount_factor * reward - current_q_value)
            self.state_value[st] = reward
            
    def move_reward(self, game: 'Game', from_pos: tuple[int, int], slide: 'Move') -> tuple[Literal[-1, 1], bool]:
        # play a move
        acceptable = game.make_move(from_pos, slide)
        # give a negative reward to the agent
        reward = -1
        # if the move is acceptable
        if acceptable:
            # give a positive reward to the agent
            reward = 1
        return reward, acceptable
    
    def game_reward(self, player: 'RLPlayer')-> Literal[-10, 0, 10]:
        if self == player:
            return 10
        else:
            return -10
    
    def train(self) -> None:
        
        all_rewards = []
        # define how many episodes to run
        pbar = trange(self.epochs)
        # define the players
        players = (self, self.opponent)
        
        for epochs in pbar:
            game = Game()
            rewards = 0
            winner = -1
            players = (players[1], players[0])
            player_idx = 1
            
            while winner < 0:
                # change player
                player_idx = (player_idx + 1) % 2
                player = players[player_idx]
                game.switch_player()
                
                ok = False
                if self == player:
                    while not ok:
                        from_pos, slide = self.choose_action(game)
                        
                        ok = game.make_move(from_pos, slide)
                        
                        #reward, ok = self.move_reward(game, from_pos, slide) 
                        state_after_move = game.convert_matrix_board_to_tuple(game.get_board())
                        
                        self.add_state(state_after_move)
                        
                        #update the action-value function
                        #self.update_state_value_table(state_after_move, reward)
                        # update the rewards
                        #rewards += reward
                else:
                    while not ok:
                        from_pos, slide = player.choose_action(game)
                        ok = game.make_move(from_pos, slide)
                winner = game.check_winner()
            
            # update the exploration rate
            self.exploration_rate = np.clip(
                np.exp(-self.exploration_decay_rate * epochs), self.min_exploration_rate, 1
            )
            
            reward = self.game_reward(player)
            #self.update_state_value_table(state_after_move, reward)
            self.update_state_value_table2(reward)
            rewards += reward
            self.reset()
            all_rewards.append(rewards)
            pbar.set_description(f'rewards value: {rewards}, current exploration rate: {self.exploration_rate:2f}')
            
        print(f'** Last 1_000 episodes - Mean rewards value: {sum(all_rewards[-1_000:]) / 1_000:.2f} **')
        print(f'** Last rewards value: {all_rewards[-1]:} **')

    def save_policy(self, name):
        fw = open(name, 'wb')
        pickle.dump(self.state_value, fw)
        fw.close()

    def load_policy(self, file):
        fr = open(file, 'rb')
        self.state_value = pickle.load(fr)
        fr.close()

# create the Q-learning player
q_learning_rl_agent = RLPlayer(
    epochs=500000,
    alpha=0.1,
    discount_factor=0.95,
    min_exploration_rate=0.01,
    exploration_decay_rate=1e-5,
    opponent=RandomPlayer(),
)
# train the Q-learning player
q_learning_rl_agent.train()

In [ ]:
def test(rl_player: 'RLPlayer', random_player, num_games):
    g = Game()
    player2_wins = 0
    games = 0
    for _ in range(num_games):
        winner = g.play(rl_player, random_player)
        games += 1
        print(games)
        g.reset()
        if winner == 0:
            player2_wins += 1

    print(f"RLPlayer won {player2_wins / num_games * 100}%")

test(q_learning_rl_agent, RandomPlayer(), 1000)

### RL Player results

- RL Player 1
    - ``epochs`` = 350000,
    - ``alpha`` = 0.1,
    - ``discount_factor`` = 0.95,
    - ``min_exploration_rate`` = 0.01,
    - ``exploration_decay_rate`` = 1e-5,
     
    **Results**
    - Last 1000 episodes - Mean rewards value: 5.08
    - win rate vs ``RandomPlayer`` in 1000 games - 79%

- RL Player 2
    - ``epochs`` = 500000,
    - ``alpha`` = 0.1,
    - ``discount_factor`` = 0.95,
    - ``min_exploration_rate`` = 0.01,
    - ``exploration_decay_rate`` = 1e-5,
     
    **Results**
    - Last 1000 episodes - Mean rewards value: 4.42
    - win rate vs ``RandomPlayer`` in 1000 games - 82%

- RL Player 3
    - ``epochs`` = 500000,
    - ``alpha`` = 0.1,
    - ``discount_factor`` = 0.95,
    - ``min_exploration_rate`` = 0.01,
    - ``exploration_decay_rate`` = 5e-6,
     
    **Results**
    - Last 1000 episodes - Mean rewards value: 1.72
    - win rate vs ``RandomPlayer`` in 1000 games - 72%
    